# Dimension Profiling – OLAP Schema Validation

**Mục tiêu:**
- Tạo các tổ hợp DISTINCT theo grain từng Dimension.
- Đếm số dòng thực tế (Distinct rows).
- Kiểm tra giá trị Unknown / Not known / Missing.

**Nguồn dữ liệu:** `Data/Processed/` (clean data, không đọc Raw)

**Schema tham chiếu:** `docs/schema_transform_summary.md`

---

## Công thức tính số dòng Dimension (Lý thuyết)

```
Distinct Rows  = số tổ hợp (DISTINCT) của các cột grain xuất hiện trong data
Unknown Rows   = số dòng trong Distinct có ít nhất 1 cột là Unknown/Not known/Missing
Expected Rows  = Distinct Rows        (nếu Unknown Rows > 0, tức surrogate Unknown đã có trong data)
               = Distinct Rows + 1   (nếu Unknown Rows = 0, cần thêm 1 dòng Unknown surrogate theo chuẩn DW)
```

**Ghi chú về từng loại Dimension:**

| Loại | Công thức ước tính lý thuyết |
|---|---|
| Lookup 1 cột (DimVehicleType, DimSeverity...) | = số giá trị distinct của cột đó |
| Combo nhiều cột (DimEnvironment, DimJunction...) | = tích số giá trị từng cột (tối đa lý thuyết), thực tế ít hơn vì không phải combo nào cũng xuất hiện |
| Bridge/Intermediate (DimVehicleInvolvement) | = số dòng của bảng nguồn (vehicles), vì mỗi dòng là 1 vehicle event duy nhất |
| DimDate | = số ngày trong khoảng thời gian dữ liệu (2021–2025 ≈ 1826 ngày) |
| DimTime | = số giá trị HH:MM duy nhất, tối đa 1440 (24h × 60 phút) |
| DimLocation | = số collision (mỗi collision có 1 địa điểm duy nhất) |

---

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path(r'd:/OLAP')
PROC_DIR = BASE_DIR / 'Data' / 'Processed'

print('Loading data...')
col = pd.read_csv(PROC_DIR / 'collisions_clean.csv', dtype=str, low_memory=False)
veh = pd.read_csv(PROC_DIR / 'vehicles_clean.csv',   dtype=str, low_memory=False)
cas = pd.read_csv(PROC_DIR / 'casualties_clean.csv', dtype=str, low_memory=False)

col['speed_limit_mph'] = pd.to_numeric(col['speed_limit_mph'], errors='coerce')

print(f'collisions : {len(col):10,} rows')
print(f'vehicles   : {len(veh):10,} rows')
print(f'casualties : {len(cas):10,} rows')


Loading data...
collisions :    513,801 rows
vehicles   :    937,265 rows
casualties :    652,821 rows


---
## Hàm tiện ích

In [2]:
UNKNOWN_PATTERNS = [
    'unknown', 'not known', 'missing', 'data missing', 'undefined',
    'not applicable', 'nan', 'none', ''
]

def count_unknown_rows(df, cols):
    """Đếm số dòng có ít nhất 1 cột thuộc nhóm Unknown."""
    mask = pd.Series([False] * len(df), index=df.index)
    for c in cols:
        if c in df.columns:
            s = df[c].fillna('').astype(str).str.lower().str.strip()
            mask = mask | s.isin(UNKNOWN_PATTERNS)
    return int(mask.sum())

def distinct_grain(df, cols):
    """Trả về DataFrame chứa các tổ hợp DISTINCT của grain cols."""
    return df[cols].drop_duplicates()

def profile_dim(name, df_source, grain_cols, formula_note=''):
    """
    Profile một Dimension.
    formula_note: công thức / giải thích cách tính số dòng của Dimension này.
    """
    distinct_df   = distinct_grain(df_source, grain_cols)
    distinct_rows = len(distinct_df)
    unknown_rows  = count_unknown_rows(distinct_df, grain_cols)
    # Expected = Distinct nếu Unknown surrogate đã tồn tại trong data;
    # ngược lại cần +1 để dành chỗ cho surrogate row 'Unknown'.
    expected_rows = distinct_rows if unknown_rows > 0 else distinct_rows + 1
    return {
        'Dimension':     name,
        'Grain Columns': ', '.join(grain_cols),
        'Distinct Rows': distinct_rows,
        'Unknown Rows':  unknown_rows,
        'Expected Rows': expected_rows,
        'Formula / Note': formula_note
    }

results = []
print('Utility functions loaded.')


Utility functions loaded.


---
## 1. DimEnvironment
**Grain:** `(light_conditions, weather_conditions, road_surface_conditions)`  
**Nguồn:** `collisions_clean.csv`  

**Công thức:**  
`Distinct Rows = số tổ hợp 3 cột xuất hiện thực tế trong collisions`  
*(Tối đa lý thuyết = N(light) × N(weather) × N(road_surface) – nhưng không phải combo nào cũng có trong data)*

In [3]:
env_grain = ['light_conditions', 'weather_conditions', 'road_surface_conditions']
env_distinct = distinct_grain(col, env_grain)

# Tối đa lý thuyết = tích số giá trị của từng cột
n_light   = col['light_conditions'].nunique()
n_weather = col['weather_conditions'].nunique()
n_road    = col['road_surface_conditions'].nunique()
theoretical_max = n_light * n_weather * n_road

r = profile_dim(
    'DimEnvironment', col, env_grain,
    formula_note=(
        f'Toi da ly thuyet = {n_light} light x {n_weather} weather x {n_road} road_surface'
        f' = {theoretical_max:,} combos. Thuc te it hon vi khong phai combo nao cung ton tai.'
    )
)
results.append(r)
print(f"DimEnvironment – distinct combos: {r['Distinct Rows']:,}")
print(f"Theoretical max (full cross-product): {theoretical_max:,}")
print(f"Unknown rows (trong distinct): {r['Unknown Rows']}")
print('\nSample distinct combinations (top 10):')
display(env_distinct.sort_values(env_grain).head(10))


DimEnvironment – distinct combos: 228
Theoretical max (full cross-product): 270
Unknown rows (trong distinct): 88

Sample distinct combinations (top 10):


,light_conditions,weather_conditions,road_surface_conditions
327,Darkness - lights lit,Fine + high winds,Dry
145008,Darkness - lights lit,Fine + high winds,Flood over 3cm. deep
3284,Darkness - lights lit,Fine + high winds,Frost or ice
187606,Darkness - lights lit,Fine + high winds,Snow
133523,Darkness - lights lit,Fine + high winds,Unknown
1063,Darkness - lights lit,Fine + high winds,Wet or damp
0,Darkness - lights lit,Fine no high winds,Dry
53100,Darkness - lights lit,Fine no high winds,Flood over 3cm. deep
204,Darkness - lights lit,Fine no high winds,Frost or ice
10022,Darkness - lights lit,Fine no high winds,Snow


---
## 2. DimRoad
**Grain:** `(road_type, speed_limit_mph, area_type)`  
**Nguồn:** `collisions_clean.csv`  
**Lưu ý:** `first_road_class` thuộc bảng con `DimRoadClass`, không phải grain DimRoad chính.

**Công thức:**  
`Distinct Rows = số tổ hợp (road_type × speed_limit_mph × area_type) thực tế`  
*(speed_limit là tập hữu hạn: 20, 30, 40, 50, 60, 70 mph – không phải số liên tục)*

In [4]:
road_grain = ['road_type', 'speed_limit_mph', 'area_type']
road_distinct = distinct_grain(col, road_grain)

n_road_type  = col['road_type'].nunique()
n_speed      = col['speed_limit_mph'].nunique()
n_area       = col['area_type'].nunique()
theoretical_max = n_road_type * n_speed * n_area

r = profile_dim(
    'DimRoad', col, road_grain,
    formula_note=(
        f'Toi da ly thuyet = {n_road_type} road_type x {n_speed} speed_limit x {n_area} area_type'
        f' = {theoretical_max:,} combos.'
    )
)
results.append(r)
print(f"DimRoad – distinct combos: {r['Distinct Rows']:,}")
print(f"Theoretical max (full cross-product): {theoretical_max:,}")
print(f"Unknown rows (trong distinct): {r['Unknown Rows']}")
print('\nAll distinct DimRoad combinations:')
display(road_distinct.sort_values(road_grain).reset_index(drop=True))


DimRoad – distinct combos: 89
Theoretical max (full cross-product): 144
Unknown rows (trong distinct): 20

All distinct DimRoad combinations:


,road_type,speed_limit_mph,area_type
0,Dual carriageway,20.0,Rural
1,Dual carriageway,20.0,Unallocated
2,Dual carriageway,20.0,Unknown
3,Dual carriageway,20.0,Urban
4,Dual carriageway,30.0,Rural
...,...,...,...
84,Unknown,60.0,Rural
85,Unknown,60.0,Urban
86,Unknown,70.0,Rural
87,Unknown,70.0,Urban


In [5]:
# DimRoadClass (bang con cua DimRoad)
# Cong thuc: Distinct Rows = so gia tri road class duy nhat
roadclass_grain = ['first_road_class']
r_rc = profile_dim(
    'DimRoadClass (sub-table)', col, roadclass_grain,
    formula_note='Bang con. Distinct Rows = so gia tri distinct cua first_road_class.'
)
results.append(r_rc)
print(f"DimRoadClass – distinct values: {r_rc['Distinct Rows']}")
display(col[['first_road_class']].drop_duplicates().sort_values('first_road_class').reset_index(drop=True))


DimRoadClass – distinct values: 7


,first_road_class
0,A
1,A(M)
2,B
3,C
4,Motorway
5,Unclassified
6,Unknown


---
## 3. DimJunction
**Grain:** `(junction_detail, junction_control, pedestrian_crossing)`  
**Nguồn:** `collisions_clean.csv`  

**Công thức:**  
`Distinct Rows = số tổ hợp 3 cột thực tế`  
*(Không phải mọi combo đều logic – vd: Not at junction thì junction_control không áp dụng → nhiều dòng Unknown)*

In [6]:
junc_grain = ['junction_detail', 'junction_control', 'pedestrian_crossing']
junc_distinct = distinct_grain(col, junc_grain)

n_jd = col['junction_detail'].nunique()
n_jc = col['junction_control'].nunique()
n_pc = col['pedestrian_crossing'].nunique()
theoretical_max = n_jd * n_jc * n_pc

r = profile_dim(
    'DimJunction', col, junc_grain,
    formula_note=(
        f'Toi da ly thuyet = {n_jd} junction_detail x {n_jc} junction_control x {n_pc} ped_crossing'
        f' = {theoretical_max:,} combos. Nhieu combo khong xuat hien vi rang buoc logic.'
    )
)
results.append(r)
print(f"DimJunction – distinct combos: {r['Distinct Rows']:,}")
print(f"Theoretical max (full cross-product): {theoretical_max:,}")
print(f"Unknown rows (trong distinct): {r['Unknown Rows']}")
print('\nSample distinct combinations (top 10):')
display(junc_distinct.sort_values(junc_grain).head(10))


DimJunction – distinct combos: 295
Theoretical max (full cross-product): 315
Unknown rows (trong distinct): 116

Sample distinct combinations (top 10):


,junction_detail,junction_control,pedestrian_crossing
50267,Crossroads,Authorised person,Central refuge - no other controls
7302,Crossroads,Authorised person,Human crossing control by other authorised person
4501,Crossroads,Authorised person,Human crossing control by school crossing patrol
853,Crossroads,Authorised person,No physical crossing facility within 50m
6133,Crossroads,Authorised person,Pedestrian light crossing (pelican or puffin o...
3305,Crossroads,Authorised person,Pedestrian phase at traffic signal
446708,Crossroads,Authorised person,Unknown
13346,Crossroads,Authorised person,Zebra crossing
177,Crossroads,Auto traffic signal,Central refuge - no other controls
9022,Crossroads,Auto traffic signal,Footbridge or subway


---
## 4. DimCasualtyProfile
**Grain:** `(casualty_class, casualty_sex, casualty_type, pedestrian_location, pedestrian_movement)`  
**Nguồn:** `casualties_clean.csv`  
**Lưu ý:** `casualty_age_band` tách sang bảng con `DimAgeBand`. `casualty_age` (số liên tục) là Measure trong Fact, không vào Dimension.

**Công thức:**  
`Distinct Rows = số tổ hợp 5 cột thực tế`  
*(pedestrian_location / pedestrian_movement chỉ có ý nghĩa với pedestrian type → combo thực tế ít hơn lý thuyết nhiều)*

In [7]:
cas_profile_grain = [
    'casualty_class', 'casualty_sex', 'casualty_type',
    'pedestrian_location', 'pedestrian_movement'
]
cas_profile_distinct = distinct_grain(cas, cas_profile_grain)

n_class = cas['casualty_class'].nunique()
n_sex   = cas['casualty_sex'].nunique()
n_type  = cas['casualty_type'].nunique()
n_ploc  = cas['pedestrian_location'].nunique()
n_pmov  = cas['pedestrian_movement'].nunique()
theoretical_max = n_class * n_sex * n_type * n_ploc * n_pmov

r = profile_dim(
    'DimCasualtyProfile', cas, cas_profile_grain,
    formula_note=(
        f'Toi da ly thuyet = {n_class} class x {n_sex} sex x {n_type} type'
        f' x {n_ploc} ped_loc x {n_pmov} ped_mov = {theoretical_max:,} combos.'
        f' Thuc te it hon vi ped_location/movement chi apply cho pedestrian type.'
    )
)
results.append(r)
print(f"DimCasualtyProfile – distinct combos: {r['Distinct Rows']:,}")
print(f"Theoretical max (full cross-product): {theoretical_max:,}")
print(f"Unknown rows (trong distinct): {r['Unknown Rows']}")
print('\nSample distinct combinations (top 15):')
display(cas_profile_distinct.sort_values(cas_profile_grain).head(15))


DimCasualtyProfile – distinct combos: 312
Theoretical max (full cross-product): 19,800
Unknown rows (trong distinct): 98

Sample distinct combinations (top 15):


,casualty_class,casualty_sex,casualty_type,pedestrian_location,pedestrian_movement
5073,Driver or rider,Female,Agricultural vehicle occupant,Not a Pedestrian,Not a Pedestrian
5057,Driver or rider,Female,Bus or coach occupant (17 or more pass seats),Not a Pedestrian,Not a Pedestrian
608403,Driver or rider,Female,Car occupant,Crossing on pedestrian crossing facility,Crossing from driver's nearside
388179,Driver or rider,Female,Car occupant,Not a Pedestrian,Crossing from driver's nearside
2175,Driver or rider,Female,Car occupant,Not a Pedestrian,Not a Pedestrian
169730,Driver or rider,Female,Car occupant,Unknown,Not a Pedestrian
1389,Driver or rider,Female,Cyclist,"In centre of carriageway - not on refuge, isla...",Crossing from driver's offside
771,Driver or rider,Female,Cyclist,Not a Pedestrian,Not a Pedestrian
5251,Driver or rider,Female,Electric motorcycle rider or passenger,Not a Pedestrian,Not a Pedestrian
5231,Driver or rider,Female,Goods vehicle (7.5 tonnes mgw and over) occupant,Not a Pedestrian,Not a Pedestrian


In [8]:
# DimAgeBand – bang con dung chung cho Casualty va Driver
# Cong thuc: Distinct Rows = so band duy nhat (7-0, 8-15, 16-20, 21-25...)
r_ab = profile_dim(
    'DimAgeBand (shared, from casualties)', cas, ['casualty_age_band'],
    formula_note='Bang con dung chung. Distinct Rows = so age band duy nhat.'
)
results.append(r_ab)

r_ab_drv = profile_dim(
    'DimAgeBand (from vehicles/driver)', veh, ['driver_age_band'],
    formula_note='Cross-check voi casualty_age_band – hai nguon phai co cung tap gia tri.'
)
results.append(r_ab_drv)

print('Casualty age bands:')
display(cas[['casualty_age_band']].drop_duplicates().sort_values('casualty_age_band').reset_index(drop=True))
print('\nDriver age bands:')
display(veh[['driver_age_band']].drop_duplicates().sort_values('driver_age_band').reset_index(drop=True))


Casualty age bands:


,casualty_age_band
0,0 - 5
1,11 - 15
2,16 - 20
3,21 - 25
4,26 - 35
5,36 - 45
6,46 - 55
7,56 - 65
8,6 - 10
9,66 - 75



Driver age bands:


,driver_age_band
0,0 - 5
1,11 - 15
2,16 - 20
3,21 - 25
4,26 - 35
5,36 - 45
6,46 - 55
7,56 - 65
8,6 - 10
9,66 - 75


---
## 5. DimVehicleEvent
**Grain:** `(vehicle_manoeuvre, skidding_and_overturning, vehicle_leaving_carriageway)`  
**Nguồn:** `vehicles_clean.csv`  

**Công thức:**  
`Distinct Rows = số tổ hợp 3 cột thực tế`  
*(Manoeuvre có nhiều giá trị → tổ hợp có thể lớn. Nếu số dòng quá lớn, xem xét tách vehicle_manoeuvre thành lookup riêng)*

In [9]:
vevent_grain = ['vehicle_manoeuvre', 'skidding_and_overturning', 'vehicle_leaving_carriageway']
vevent_distinct = distinct_grain(veh, vevent_grain)

n_man  = veh['vehicle_manoeuvre'].nunique()
n_skid = veh['skidding_and_overturning'].nunique()
n_lc   = veh['vehicle_leaving_carriageway'].nunique()
theoretical_max = n_man * n_skid * n_lc

r = profile_dim(
    'DimVehicleEvent', veh, vevent_grain,
    formula_note=(
        f'Toi da ly thuyet = {n_man} manoeuvre x {n_skid} skidding x {n_lc} leaving'
        f' = {theoretical_max:,} combos.'
    )
)
results.append(r)
print(f"DimVehicleEvent – distinct combos: {r['Distinct Rows']:,}")
print(f"Theoretical max (full cross-product): {theoretical_max:,}")
print(f"Unknown rows (trong distinct): {r['Unknown Rows']}")
print('\nSample distinct combinations (top 15):')
display(vevent_distinct.sort_values(vevent_grain).head(15))


DimVehicleEvent – distinct combos: 718
Theoretical max (full cross-product): 1,080
Unknown rows (trong distinct): 327

Sample distinct combinations (top 15):


,vehicle_manoeuvre,skidding_and_overturning,vehicle_leaving_carriageway
5787,Changing lane to left,Overturned,Did not leave carriageway
743,Changing lane to left,Overturned,Nearside
28964,Changing lane to left,Overturned,Nearside and rebounded
25615,Changing lane to left,Overturned,Offside
14533,Changing lane to left,Overturned,Offside - crossed central reservation
221651,Changing lane to left,Overturned,Offside and rebounded
52644,Changing lane to left,Overturned,Offside on to central reservation
454193,Changing lane to left,Overturned,Offside on to centrl res + rebounded
903525,Changing lane to left,Overturned,Straight ahead at junction
645,Changing lane to left,Skidded,Did not leave carriageway


---
## 6. DimVehicleType
**Grain:** `(vehicle_type)` | **Nguồn:** `vehicles_clean.csv`

**Công thức:** `Distinct Rows = số giá trị vehicle_type duy nhất`

In [10]:
r = profile_dim(
    'DimVehicleType', veh, ['vehicle_type'],
    formula_note='Distinct Rows = so gia tri vehicle_type duy nhat.'
)
results.append(r)
print(f"DimVehicleType – distinct values: {r['Distinct Rows']}")
display(veh[['vehicle_type']].drop_duplicates().sort_values('vehicle_type').reset_index(drop=True))


DimVehicleType – distinct values: 19


,vehicle_type
0,Agricultural vehicle
1,Bus or coach (17 or more pass seats)
2,Car
3,Electric motorcycle
4,Goods 7.5 tonnes mgw and over
5,Goods over 3.5t. and under 7.5t
6,Minibus (8 - 16 passenger seats)
7,Mobility scooter
8,Motorcycle 125cc and under
9,Motorcycle 50cc and under


---
## 7. DimDriverProfile
**Grain:** `(driver_sex)` | **Nguồn:** `vehicles_clean.csv`

**Công thức:** `Distinct Rows = số giá trị driver_sex duy nhất`  
*(driver_age_band tách sang DimAgeBand)*

In [11]:
r = profile_dim(
    'DimDriverProfile', veh, ['driver_sex'],
    formula_note='Distinct Rows = so gia tri driver_sex duy nhat. Age band tach sang DimAgeBand.'
)
results.append(r)
print(f"DimDriverProfile – distinct values: {r['Distinct Rows']}")
display(veh[['driver_sex']].drop_duplicates().reset_index(drop=True))


DimDriverProfile – distinct values: 3


,driver_sex
0,Male
1,Female
2,Unknown


---
## 8. DimSeverity (Role-playing)
**Union** `collision_severity` + `casualty_severity`  
**Vai trò:** 2 FK trong FactCasualty: `CasualtySeverityKey` và `CollisionSeverityKey`.

**Công thức:** `Distinct Rows = union các giá trị của collision_severity và casualty_severity`  
*(Hai cột phải dùng chung cùng 1 bảng vì cùng miền giá trị: Fatal / Serious / Slight)*

In [12]:
col_sev = col[['collision_severity']].rename(columns={'collision_severity': 'severity_label'})
cas_sev = cas[['casualty_severity']].rename(columns={'casualty_severity': 'severity_label'})
all_sev = pd.concat([col_sev, cas_sev], ignore_index=True)
r = profile_dim(
    'DimSeverity (role-playing)', all_sev, ['severity_label'],
    formula_note='Distinct Rows = union(collision_severity, casualty_severity). Hai cot phai co cung mien gia tri.'
)
results.append(r)
print(f"DimSeverity – distinct values: {r['Distinct Rows']}")
print(f"Unknown rows: {r['Unknown Rows']}")
display(all_sev.drop_duplicates().sort_values('severity_label').reset_index(drop=True))


DimSeverity – distinct values: 3
Unknown rows: 0


,severity_label
0,Fatal
1,Serious
2,Slight


---
## 9. DimVehicleInvolvement (Bridge / Intermediate table)
**Grain:** `(collision_id, vehicle_reference)` – khóa tự nhiên của bảng vehicles

**Công thức:** `Distinct Rows = số dòng trong vehicles_clean`  
*(Vì mỗi dòng vehicles đã là 1 vehicle-in-collision duy nhất. Nếu Distinct < Total → có duplicate key)*

In [13]:
vinv_grain = ['collision_id', 'vehicle_reference']
vinv_distinct = distinct_grain(veh, vinv_grain)
r = profile_dim(
    'DimVehicleInvolvement', veh, vinv_grain,
    formula_note='Distinct Rows = so dong vehicles_clean. Neu Distinct < Total thi co duplicate key.'
)
results.append(r)
print(f"DimVehicleInvolvement – distinct keys : {r['Distinct Rows']:,}")
print(f"vehicles_clean total rows             : {len(veh):,}")
ok = r['Distinct Rows'] == len(veh)
print(f"Duplicate check: {'OK - No duplicate keys' if ok else 'DUPLICATE KEYS FOUND - check source data'}")


DimVehicleInvolvement – distinct keys : 937,265
vehicles_clean total rows             : 937,265
Duplicate check: OK - No duplicate keys


---
## 10. DimDate & DimTime

**Công thức DimDate:** `Distinct Rows = số ngày duy nhất trong collision_date`  
*(Thực tế nên dùng Calendar Table đầy đủ = 365 × số năm = 1826 ngày cho 2021-2025)*

**Công thức DimTime:** `Distinct Rows = số giá trị HH:MM duy nhất, tối đa 1440 (24×60)`

In [14]:
# DimDate
r_date = profile_dim(
    'DimDate', col, ['collision_date'],
    formula_note='Distinct Rows = so ngay duy nhat trong data. Calendar table day du = 365 x so nam.'
)
results.append(r_date)

# DimTime
r_time = profile_dim(
    'DimTime', col, ['collision_time'],
    formula_note='Distinct Rows = so gia tri HH:MM duy nhat. Toi da ly thuyet = 24x60 = 1440.'
)
results.append(r_time)

print(f"DimDate  – distinct dates : {r_date['Distinct Rows']:,}  (calendar full: ~1826 ngay)")
print(f"DimTime  – distinct times : {r_time['Distinct Rows']:,}  (theoretical max: 1440)")
print(f"Date range: {col['collision_date'].min()} -> {col['collision_date'].max()}")


DimDate  – distinct dates : 1,826  (calendar full: ~1826 ngay)
DimTime  – distinct times : 1,440  (theoretical max: 1440)
Date range: 2021-01-01 -> 2025-12-31


---
## 11. DimLocation & DimLocalAuthority

**Công thức DimLocalAuthority:** `Distinct Rows = số mã ONS district duy nhất`  

**Công thức DimLocation:** `Distinct Rows = số collision_id duy nhất`  
*(Mỗi collision có đúng 1 địa điểm → Distinct phải bằng tổng số dòng collisions)*

In [15]:
# DimLocalAuthority
r_la = profile_dim(
    'DimLocalAuthority', col, ['local_authority_district_code'],
    formula_note='Distinct Rows = so ma ONS district duy nhat trong data.'
)
results.append(r_la)

# DimLocation
loc_grain = ['collision_id', 'longitude', 'latitude', 'local_authority_district_code']
r_loc = profile_dim(
    'DimLocation', col, loc_grain,
    formula_note='Distinct Rows phai bang so collision (1 collision = 1 location). Neu khac -> check null toa do.'
)
results.append(r_loc)

print(f"DimLocalAuthority – distinct LAs  : {r_la['Distinct Rows']:,}")
print(f"DimLocation       – distinct rows  : {r_loc['Distinct Rows']:,}")
print(f"collisions total                   : {len(col):,}")
ok = r_loc['Distinct Rows'] == len(col)
print(f"Location = Collision count: {'OK' if ok else 'CHECK - co null toa do hoac dup collision_id'}")


DimLocalAuthority – distinct LAs  : 370
DimLocation       – distinct rows  : 513,801
collisions total                   : 513,801
Location = Collision count: OK


---
## 12. FactCasualty – kiểm tra grain

**Grain:** `(collision_id, casualty_reference)`  
**Công thức:** `Distinct Rows = Tổng số dòng casualties_clean`  
*(Nếu Distinct < Total → có duplicate key → vi phạm grain → cần kiểm tra lại nguồn)*

In [16]:
fact_grain    = ['collision_id', 'casualty_reference']
fact_distinct = distinct_grain(cas, fact_grain)
print('FactCasualty Grain Check')
print(f'  Total rows in casualties_clean             : {len(cas):,}')
print(f'  Distinct (collision_id, casualty_reference): {len(fact_distinct):,}')
print(f'  Duplicates                                 : {len(cas) - len(fact_distinct):,}')
ok = len(fact_distinct) == len(cas)
print(f"  Status: {'[OK] No duplicates - grain is clean' if ok else '[!!] DUPLICATE KEYS - grain violation!'}")


FactCasualty Grain Check
  Total rows in casualties_clean             : 652,821
  Distinct (collision_id, casualty_reference): 652,821
  Duplicates                                 : 0
  Status: [OK] No duplicates - grain is clean


---
## Báo Cáo Tổng Hợp – Dimension Profile Summary

In [18]:
df_report = pd.DataFrame(results)

summary = df_report[['Dimension', 'Distinct Rows', 'Unknown Rows', 'Expected Rows']].copy()

display(
    summary.style
    .format({'Distinct Rows': '{:,}', 'Unknown Rows': '{:,}', 'Expected Rows': '{:,}'})
    .set_caption('Dimension Profile Summary – OLAP Schema Validation')
    .set_table_styles([{'selector': 'caption', 'props': [('font-weight','bold'),('font-size','14px')]}])
)


,Dimension,Distinct Rows,Unknown Rows,Expected Rows
0,DimEnvironment,228,88,228
1,DimRoad,89,20,89
2,DimRoadClass (sub-table),7,1,7
3,DimJunction,295,116,295
4,DimCasualtyProfile,312,98,312
5,"DimAgeBand (shared, from casualties)",12,1,12
6,DimAgeBand (from vehicles/driver),12,1,12
7,DimVehicleEvent,718,327,718
8,DimVehicleType,19,1,19
9,DimDriverProfile,3,1,3


---
## 🔍 Phân Tích Unknown Values theo từng cột

In [19]:
print('=' * 70)
print('UNKNOWN VALUE BREAKDOWN BY DIMENSION')
print('=' * 70)

def print_unknown_breakdown(label, df, cols):
    print(f'\n  {label}')
    print('  ' + '-' * 66)
    for c in cols:
        if c not in df.columns: continue
        s   = df[c].fillna('').astype(str).str.lower().str.strip()
        n   = s.isin(UNKNOWN_PATTERNS).sum()
        pct = n / len(df) * 100 if len(df) > 0 else 0
        print(f'  {c:<45} unknown={n:>7,} / {len(df):>8,}  ({pct:5.2f}%)')

print_unknown_breakdown('DimEnvironment   [collisions_clean]',   col, ['light_conditions','weather_conditions','road_surface_conditions'])
print_unknown_breakdown('DimRoad          [collisions_clean]',   col, ['road_type','speed_limit_mph','area_type'])
print_unknown_breakdown('DimJunction      [collisions_clean]',   col, ['junction_detail','junction_control','pedestrian_crossing'])
print_unknown_breakdown('DimCasualtyProfile [casualties_clean]', cas, ['casualty_class','casualty_sex','casualty_type','pedestrian_location','pedestrian_movement'])
print_unknown_breakdown('DimVehicleEvent  [vehicles_clean]',     veh, ['vehicle_manoeuvre','skidding_and_overturning','vehicle_leaving_carriageway'])
print('\n' + '=' * 70)


UNKNOWN VALUE BREAKDOWN BY DIMENSION

  DimEnvironment   [collisions_clean]
  ------------------------------------------------------------------
  light_conditions                              unknown=  8,630 /  513,801  ( 1.68%)
  weather_conditions                            unknown= 14,578 /  513,801  ( 2.84%)
  road_surface_conditions                       unknown= 10,248 /  513,801  ( 1.99%)

  DimRoad          [collisions_clean]
  ------------------------------------------------------------------
  road_type                                     unknown= 13,060 /  513,801  ( 2.54%)
  speed_limit_mph                               unknown=      3 /  513,801  ( 0.00%)
  area_type                                     unknown=      8 /  513,801  ( 0.00%)

  DimJunction      [collisions_clean]
  ------------------------------------------------------------------
  junction_detail                               unknown= 21,411 /  513,801  ( 4.17%)
  junction_control                          

---
## 💾 Xuất CSV báo cáo

In [20]:
OUTPUT_CSV = BASE_DIR / 'Data' / 'Profiling' / 'dimension_profile_report.csv'
export_df = df_report[[
    'Dimension', 'Grain Columns', 'Distinct Rows', 'Unknown Rows',
    'Expected Rows', 'Formula / Note'
]].copy()
export_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Bao cao da xuat: {OUTPUT_CSV}')
display(export_df)


Bao cao da xuat: d:\OLAP\Data\Profiling\dimension_profile_report.csv


,Dimension,Grain Columns,Distinct Rows,Unknown Rows,Expected Rows,Formula / Note
0,DimEnvironment,"light_conditions, weather_conditions, road_sur...",228,88,228,Toi da ly thuyet = 5 light x 9 weather x 6 roa...
1,DimRoad,"road_type, speed_limit_mph, area_type",89,20,89,Toi da ly thuyet = 6 road_type x 6 speed_limit...
2,DimRoadClass (sub-table),first_road_class,7,1,7,Bang con. Distinct Rows = so gia tri distinct ...
3,DimJunction,"junction_detail, junction_control, pedestrian_...",295,116,295,Toi da ly thuyet = 7 junction_detail x 5 junct...
4,DimCasualtyProfile,"casualty_class, casualty_sex, casualty_type, p...",312,98,312,Toi da ly thuyet = 3 class x 3 sex x 20 type x...
5,"DimAgeBand (shared, from casualties)",casualty_age_band,12,1,12,Bang con dung chung. Distinct Rows = so age ba...
6,DimAgeBand (from vehicles/driver),driver_age_band,12,1,12,Cross-check voi casualty_age_band – hai nguon ...
7,DimVehicleEvent,"vehicle_manoeuvre, skidding_and_overturning, v...",718,327,718,Toi da ly thuyet = 18 manoeuvre x 6 skidding x...
8,DimVehicleType,vehicle_type,19,1,19,Distinct Rows = so gia tri vehicle_type duy nhat.
9,DimDriverProfile,driver_sex,3,1,3,Distinct Rows = so gia tri driver_sex duy nhat...
